# NIFTY 500 ATH Breakout Alert — Local Test Notebook

Use this notebook to test the pipeline end-to-end on your machine before deploying to AWS Lambda.

**What this notebook does:**
1. Installs required dependencies
2. Lets you set credentials locally (never committed to GitHub)
3. Tests each function individually
4. Runs the full pipeline
5. Previews the email output without actually sending it

---
> ⚠️ **Never paste your real email password directly into a notebook cell.** Use the `getpass` method in Cell 3 — it prompts you securely without displaying or saving the value.

## Cell 1 — Install Dependencies

In [ ]:
# Run this cell once to install all required packages
# If already installed, this will just confirm versions
%pip install pandas yfinance requests beautifulsoup4 --quiet
print("All dependencies installed.")

## Cell 2 — Imports

In [ ]:
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import os
import json
import datetime
import ssl
import smtplib
from email.message import EmailMessage
from getpass import getpass

print("Imports successful.")

## Cell 3 — Set Credentials Locally

These are stored only in memory for this session. They are **not saved to disk** and **not committed to GitHub**.

For `EMAIL_PASSWORD`: use a **Gmail App Password**, not your Gmail login password.  
Generate one at: [myaccount.google.com/apppasswords](https://myaccount.google.com/apppasswords)

In [ ]:
# Enter your credentials — getpass hides the password input
os.environ['EMAIL_SENDER']   = input("Enter sender Gmail address: ")
os.environ['EMAIL_PASSWORD'] = getpass("Enter Gmail App Password: ")
os.environ['RECIPIENT']      = input("Enter recipient email address: ")

print("Credentials loaded into environment variables.")

## Cell 4 — Define Functions

Paste the full production code here. This mirrors `lambda_function.py` exactly.

In [ ]:
def ath_stock_finder(ticker):
    """
    Identifies if a stock is currently at its all-time high.

    Args:
        ticker (str): NSE ticker symbol with '.NS' suffix (e.g., 'RELIANCE.NS').

    Returns:
        tuple: (ticker_name, current_price, previous_ath) if the stock is at ATH.
        None: If the stock is not at ATH.
    """
    df = yf.download(
        ticker,
        interval='1mo',
        period="max",
        back_adjust=True,
        progress=False,
        auto_adjust=True
    )[['Close', 'High']]

    df['Date'] = pd.to_datetime(df.index).strftime('%Y-%m-%d')
    df.index = df['Date']
    df = df.drop('Date', axis=1)
    df.columns = ['Close', 'High']

    previous_ath = df['High'].shift(1).max()
    current_price = df['Close'].iloc[-1]
    ticker_name = ticker.replace('.NS', '')

    if current_price > previous_ath:
        return ticker_name, current_price, previous_ath
    return None


def nifty_500_ath():
    """
    Scrapes NIFTY 500 tickers from Wikipedia and identifies ATH breakout stocks.

    Returns:
        pd.DataFrame: ATH breakout stocks with enriched details.
        list: Tickers for which data could not be downloaded.
    """
    url = "https://en.wikipedia.org/wiki/NIFTY_500"
    headers = {"User-Agent": "CoolBot/0.0 (https://example.org/coolbot/; coolbot@example.org)"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    table = soup.find("table", {"class": "wikitable sortable mw-collapsible"})

    if not table:
        print("Could not find the NIFTY 500 table on the Wikipedia page.")
        return pd.DataFrame(), []

    rows = table.find_all("tr")
    col_headers = [header.text.strip() for header in rows[0].find_all("td")]

    data = []
    for row in rows[1:]:
        cells = row.find_all("td")
        if cells:
            data.append([cell.text.strip() for cell in cells])

    nifty_500_df = pd.DataFrame(data, columns=col_headers)
    nifty_500_df.rename(
        columns={
            "Symbol": "Ticker",
            "Company  Name": "Company Name",
            "ISIN  Code": "ISIN Code"
        },
        inplace=True
    )
    print(f"Scraped {len(nifty_500_df)} tickers from Wikipedia.")

    nifty_500_tickers = nifty_500_df['Ticker'] + '.NS'

    ath_stocks_list = []
    failed_tickers = []

    for ticker in nifty_500_tickers:
        try:
            result = ath_stock_finder(ticker)
            if result:
                ath_stocks_list.append(result)
        except Exception as e:
            failed_tickers.append(ticker.replace('.NS', ''))
            print(f"Failed to fetch data for {ticker}: {e}")

    print(f"Scan complete. {len(ath_stocks_list)} ATH breakouts found. {len(failed_tickers)} tickers failed.")

    ath_stocks = pd.DataFrame(ath_stocks_list, columns=['Ticker', 'Current Price', 'Previous ATH'])
    ath_stocks = ath_stocks.join(nifty_500_df.set_index('Ticker'), on='Ticker')
    ath_stocks.drop(['Sl.No', 'Series', 'ISIN Code'], axis=1, inplace=True)
    ath_stocks['Current Price'] = ath_stocks['Current Price'].astype(float).round(2)
    ath_stocks['Previous ATH'] = ath_stocks['Previous ATH'].astype(float).round(2)
    ath_stocks = ath_stocks[['Company Name', 'Ticker', 'Industry', 'Current Price', 'Previous ATH']]

    return ath_stocks, failed_tickers


def send_email(sender_email, sender_password, recipient, subject, df, failed_tickers):
    """
    Sends an HTML-formatted email containing the ATH breakout stock list.
    """
    msg = EmailMessage()
    msg['Subject'] = subject
    msg['From'] = sender_email
    msg['To'] = recipient

    html_table = df.to_html(index=False, justify='center')
    email_body = f"<p>List of All Time High Stocks. Keep Growing!! 🚀🚀</p>{html_table}"

    if failed_tickers:
        failed_tickers_str = ', '.join(failed_tickers)
        email_body += f"<p><em>Note: Data could not be retrieved for the following tickers: {failed_tickers_str}</em></p>"

    msg.add_alternative(email_body, subtype='html')

    context = ssl.create_default_context()
    with smtplib.SMTP_SSL('smtp.gmail.com', 465, context=context) as server:
        server.login(sender_email, sender_password)
        server.send_message(msg)

    print("Email sent successfully.")


print("All functions defined.")

## Cell 5 — Unit Test: Scraper Only

Tests that Wikipedia scraping works and the ticker list is populated correctly.  
**Does not download stock prices — runs in a few seconds.**

In [ ]:
url = "https://en.wikipedia.org/wiki/NIFTY_500"
headers = {"User-Agent": "CoolBot/0.0 (https://example.org/coolbot/; coolbot@example.org)"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")
table = soup.find("table", {"class": "wikitable sortable mw-collapsible"})

if table:
    rows = table.find_all("tr")
    col_headers = [h.text.strip() for h in rows[0].find_all("td")]
    data = [[c.text.strip() for c in row.find_all("td")] for row in rows[1:] if row.find_all("td")]
    df_test = pd.DataFrame(data, columns=col_headers)
    df_test.rename(columns={"Symbol": "Ticker", "Company  Name": "Company Name", "ISIN  Code": "ISIN Code"}, inplace=True)
    print(f"✅ Scraper working. {len(df_test)} tickers found.")
    print(f"Columns: {list(df_test.columns)}")
    display(df_test.head(10))
else:
    print("❌ Table not found. Wikipedia page structure may have changed.")

## Cell 6 — Unit Test: Single Ticker ATH Check

Tests `ath_stock_finder` on a single stock before running the full 500-ticker scan.  
Change the ticker below to any NSE stock you want to verify.

In [ ]:
# Test with a single ticker — change this to any NIFTY 500 stock
test_ticker = "RELIANCE.NS"

try:
    result = ath_stock_finder(test_ticker)
    if result:
        ticker_name, current_price, previous_ath = result
        print(f"✅ ATH Breakout Detected!")
        print(f"   Ticker        : {ticker_name}")
        print(f"   Current Price : ₹{current_price}")
        print(f"   Previous ATH  : ₹{previous_ath}")
    else:
        print(f"ℹ️  {test_ticker.replace('.NS','')} is NOT at an all-time high right now.")
except Exception as e:
    print(f"❌ Error fetching data for {test_ticker}: {e}")

## Cell 7 — Unit Test: Small Batch (10 Tickers)

Runs the ATH scan on the first 10 tickers only — fast sanity check before the full 500-ticker run.  
**Takes ~1–2 minutes.**

In [ ]:
url = "https://en.wikipedia.org/wiki/NIFTY_500"
headers = {"User-Agent": "CoolBot/0.0 (https://example.org/coolbot/; coolbot@example.org)"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")
table = soup.find("table", {"class": "wikitable sortable mw-collapsible"})
rows = table.find_all("tr")
col_headers = [h.text.strip() for h in rows[0].find_all("td")]
data = [[c.text.strip() for c in row.find_all("td")] for row in rows[1:] if row.find_all("td")]
nifty_500_df = pd.DataFrame(data, columns=col_headers)
nifty_500_df.rename(columns={"Symbol": "Ticker", "Company  Name": "Company Name", "ISIN  Code": "ISIN Code"}, inplace=True)

# Only scan first 10 tickers for this test
sample_tickers = (nifty_500_df['Ticker'] + '.NS').head(10).tolist()

ath_results = []
failed = []

for ticker in sample_tickers:
    try:
        result = ath_stock_finder(ticker)
        if result:
            ath_results.append(result)
            print(f"✅ ATH: {ticker}")
        else:
            print(f"   No breakout: {ticker}")
    except Exception as e:
        failed.append(ticker)
        print(f"❌ Failed: {ticker} — {e}")

print(f"\nBatch complete. {len(ath_results)} breakouts | {len(failed)} failed")

if ath_results:
    df_batch = pd.DataFrame(ath_results, columns=['Ticker', 'Current Price', 'Previous ATH'])
    display(df_batch)

## Cell 8 — Full Pipeline Run (All 500 Tickers)

Runs the complete `nifty_500_ath()` function across all NIFTY 500 stocks.  
⚠️ **This takes 15–30 minutes** depending on your internet speed. Run only when you want the real output.

In [ ]:
print("Starting full NIFTY 500 ATH scan...")
print("Expected duration: 15–30 minutes\n")

ath_df, failed_tickers = nifty_500_ath()

print(f"\n--- Scan Summary ---")
print(f"ATH Breakouts Found : {len(ath_df)}")
print(f"Failed Tickers      : {len(failed_tickers)}")

if not ath_df.empty:
    display(ath_df)
else:
    print("No ATH breakouts detected in this run.")

## Cell 9 — Preview Email Output (No Email Sent)

Renders the HTML email body in the notebook so you can see exactly what the recipient will receive.  
**Does not send anything.**

In [ ]:
from IPython.display import HTML

# Uses ath_df and failed_tickers from Cell 8
# If you haven't run Cell 8, create a small sample DataFrame to preview
try:
    preview_df = ath_df
except NameError:
    # Fallback sample data if full scan hasn't been run
    preview_df = pd.DataFrame({
        'Company Name': ['Reliance Industries', 'HDFC Bank', 'Infosys'],
        'Ticker': ['RELIANCE', 'HDFCBANK', 'INFY'],
        'Industry': ['Energy', 'Financial Services', 'IT'],
        'Current Price': [2987.50, 1743.20, 1892.00],
        'Previous ATH': [2950.00, 1720.00, 1875.00]
    })
    failed_tickers = ['SAMPLE1', 'SAMPLE2']
    print("ℹ️  Using sample data — run Cell 8 first for real results.")

month = datetime.datetime.now().strftime("%B %Y")
html_table = preview_df.to_html(index=False, justify='center')
email_body = f"<p>List of All Time High Stocks. Keep Growing!! 🚀🚀</p>{html_table}"

if failed_tickers:
    failed_str = ', '.join(failed_tickers)
    email_body += f"<p><em>Note: Data could not be retrieved for: {failed_str}</em></p>"

print(f"Subject: Nifty 500 All-Time High Breakout Stocks — {month}\n")
HTML(email_body)

## Cell 10 — Send Real Email

Sends the actual email using credentials from Cell 3.  
Run Cell 8 first to populate `ath_df` and `failed_tickers`.

> ✅ This replicates exactly what AWS Lambda does when triggered by EventBridge.

In [ ]:
sender_email   = os.environ['EMAIL_SENDER']
sender_password = os.environ['EMAIL_PASSWORD']
recipient      = os.environ['RECIPIENT']

month   = datetime.datetime.now().strftime("%B %Y")
subject = f"Nifty 500 All-Time High Breakout Stocks — {month}"

try:
    send_email(sender_email, sender_password, recipient, subject, ath_df, failed_tickers)
    print(f"✅ Email sent to {recipient}")
except Exception as e:
    print(f"❌ Email failed: {e}")

---

## Troubleshooting

| Problem | Fix |
|---|---|
| Wikipedia table not found | Page structure may have changed — inspect the page and update the `class` selector in `soup.find()` |
| `yfinance` returns empty DataFrame | Ticker may be delisted or the `.NS` suffix is wrong — verify on finance.yahoo.com |
| Gmail SMTP auth failure | Make sure you're using a **Gmail App Password**, not your account password. 2FA must be enabled. |
| Full scan very slow | Normal — 500 sequential API calls take time. Consider adding `time.sleep(0.1)` between calls if you hit rate limits. |
| `KeyError: EMAIL_SENDER` | You skipped Cell 3 — re-run it to load credentials into environment variables. |